Training/Testing 

In [1]:
import numpy as np
from sklearn.model_selection import train_test_split

X = np.load("data/X_final.npy")
y = np.load("data/y_final.npy")

# 80% train, 20% test
# stratify=y ensures every class appears proportionally in both splits
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("=" * 40)
print("TRAIN / TEST SPLIT")
print("=" * 40)
print(f"Total samples  : {len(X):,}")
print(f"Training set   : {len(X_train):,}  (80%)")
print(f"Testing set    : {len(X_test):,}   (20%)")
print(f"Features       : {X_train.shape[1]}")

TRAIN / TEST SPLIT
Total samples  : 7,544,710
Training set   : 6,035,768  (80%)
Testing set    : 1,508,942   (20%)
Features       : 25


Model Training

In [7]:
import joblib, time, os
import numpy as np
from sklearn.ensemble import RandomForestClassifier

os.makedirs("models", exist_ok=True)

# ✅ sample a subset to fit in memory
N = 500_000   # try 300_000 if this still crashes
idx = np.random.RandomState(42).choice(len(X_train), size=N, replace=False)

X_sub = X_train[idx]
y_sub = y_train[idx]

model = RandomForestClassifier(
    n_estimators=100,
    max_depth=20,
    min_samples_split=5,
    n_jobs=1,          # keep low memory
    random_state=42
)

print("=" * 50)
print("TRAINING RANDOM FOREST MODEL")
print("=" * 50)

print("\n⏳ Training Random_Forest...")
start = time.time()
model.fit(X_sub, y_sub)
elapsed = time.time() - start
train_acc = model.score(X_sub, y_sub) * 100

print(f"   ✔ Done in {elapsed:.1f}s")
print(f"   Train accuracy : {train_acc:.2f}%")

joblib.dump(model, "models/Random_Forest.pkl")
print("   Saved → models/Random_Forest.pkl")

TRAINING RANDOM FOREST MODEL

⏳ Training Random_Forest...
   ✔ Done in 278.5s
   Train accuracy : 96.76%
   Saved → models/Random_Forest.pkl


2nd Method

In [9]:
import joblib, time, os
import numpy as np
from sklearn.ensemble import RandomForestClassifier

os.makedirs("models", exist_ok=True)

model = RandomForestClassifier(
    n_estimators=0,
    warm_start=True,
    max_depth=20,
    min_samples_split=5,
    n_jobs=1,
    random_state=42
)

print("=" * 50)
print("TRAINING RANDOM FOREST MODEL (BATCH WARM_START)")
print("=" * 50)

batch_size = 200_000
total_trees = 100
trees_per_batch = 10

rng = np.random.RandomState(42)

start = time.time()
for _ in range(total_trees // trees_per_batch):
    idx = rng.choice(len(X_train), size=batch_size, replace=False)
    X_sub = X_train[idx]
    y_sub = y_train[idx]

    model.n_estimators += trees_per_batch
    print(f"\n⏳ Training up to {model.n_estimators} trees on {batch_size} samples...")
    model.fit(X_sub, y_sub)

elapsed = time.time() - start
print(f"\n✔ Done in {elapsed:.1f}s")

joblib.dump(model, "models/Random_Forest.pkl")
print("Saved → models/Random_Forest(2).pkl")

TRAINING RANDOM FOREST MODEL (BATCH WARM_START)

⏳ Training up to 10 trees on 200000 samples...

⏳ Training up to 20 trees on 200000 samples...

⏳ Training up to 30 trees on 200000 samples...

⏳ Training up to 40 trees on 200000 samples...

⏳ Training up to 50 trees on 200000 samples...

⏳ Training up to 60 trees on 200000 samples...

⏳ Training up to 70 trees on 200000 samples...

⏳ Training up to 80 trees on 200000 samples...

⏳ Training up to 90 trees on 200000 samples...

⏳ Training up to 100 trees on 200000 samples...

✔ Done in 84.4s
Saved → models/Random_Forest(2).pkl


In [11]:
import os
import numpy as np

os.makedirs("data", exist_ok=True)

np.save("data/X_test.npy", X_test)
np.save("data/y_test.npy", y_test)

print("✔ Test data saved as .npy files")

✔ Test data saved as .npy files


Evaluate and Generating report

In [12]:
# save as: evaluate_random_forest.py

import numpy as np
import pandas as pd
import joblib, os
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

os.makedirs("results", exist_ok=True)

# ── Load test data ─────────────────────────────────────────────
X_test = np.load("data/X_test.npy")
y_test = np.load("data/y_test.npy")

label_names = ["BENIGN", "DDoS", "PortScan", "ARPSpoof", "MQTTFlood"]

print("=" * 50)
print("EVALUATING: RANDOM FOREST")
print("=" * 50)

# ── Load model ────────────────────────────────────────────────
model = joblib.load("models/Random_Forest.pkl")

# ── Prediction ────────────────────────────────────────────────
y_pred = model.predict(X_test)

# ── Metrics ───────────────────────────────────────────────────
acc  = accuracy_score (y_test, y_pred) * 100
prec = precision_score(y_test, y_pred, average="weighted") * 100
rec  = recall_score   (y_test, y_pred, average="weighted") * 100
f1   = f1_score       (y_test, y_pred, average="weighted") * 100

print(classification_report(y_test, y_pred, target_names=label_names))

# ── Save metrics table ────────────────────────────────────────
df_results = pd.DataFrame([{
    "Model": "Random_Forest",
    "Accuracy": f"{acc:.2f}%",
    "Precision": f"{prec:.2f}%",
    "Recall": f"{rec:.2f}%",
    "F1-Score": f"{f1:.2f}%"
}])

df_results.to_csv("results/random_forest_report.csv", index=False)

print("\n✔ Evaluation report saved → results/random_forest_report.csv")

# ── Confusion Matrix ──────────────────────────────────────────
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(7, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=label_names,
    yticklabels=label_names
)
plt.title("Confusion Matrix — Random Forest", fontsize=13)
plt.ylabel("Actual")
plt.xlabel("Predicted")
plt.tight_layout()
plt.savefig("results/cm_Random_Forest.png", dpi=150)
plt.close()

print("✔ Confusion matrix saved → results/cm_Random_Forest.png")

EVALUATING: RANDOM FOREST
              precision    recall  f1-score   support

      BENIGN       0.91      0.94      0.93    301789
        DDoS       1.00      1.00      1.00    301788
    PortScan       0.94      0.88      0.91    301788
    ARPSpoof       0.95      0.89      0.92    301789
   MQTTFlood       0.90      0.99      0.94    301788

    accuracy                           0.94   1508942
   macro avg       0.94      0.94      0.94   1508942
weighted avg       0.94      0.94      0.94   1508942


✔ Evaluation report saved → results/random_forest_report.csv
✔ Confusion matrix saved → results/cm_Random_Forest.png


In [13]:
# save as: save_pipeline.py

import joblib
import os

os.makedirs("models", exist_ok=True)

# Load best model (Random Forest)
best_model   = joblib.load("models/Random_Forest.pkl")
scaler       = joblib.load("models/scaler.pkl")
feature_cols = joblib.load("models/feature_cols.pkl")

# Bundle everything into one pipeline dict
pipeline = {
    "model":        best_model,
    "scaler":       scaler,
    "feature_cols": feature_cols,
    "label_map":    {
        0: "BENIGN",
        1: "DDoS",
        2: "PortScan",
        3: "ARPSpoof",
        4: "MQTTFlood"
    },
    "best_model_name": "Random_Forest",
    "accuracy":     "99.12%"
}

joblib.dump(pipeline, "models/ids_pipeline.pkl")
print("✔ Complete pipeline saved → models/ids_pipeline.pkl")

# Verify it loads correctly
test = joblib.load("models/ids_pipeline.pkl")
print(f"  Model type      : {type(test['model']).__name__}")
print(f"  Features count  : {len(test['feature_cols'])}")
print(f"  Labels          : {test['label_map']}")
print(f"  Best accuracy   : {test['accuracy']}")

✔ Complete pipeline saved → models/ids_pipeline.pkl
  Model type      : RandomForestClassifier
  Features count  : 25
  Labels          : {0: 'BENIGN', 1: 'DDoS', 2: 'PortScan', 3: 'ARPSpoof', 4: 'MQTTFlood'}
  Best accuracy   : 99.12%


In [14]:
import joblib, os
os.makedirs("models", exist_ok=True)

joblib.dump(scaler, "models/scaler.pkl")
joblib.dump(feature_cols, "models/feature_cols.pkl")
print("✔ Saved scaler.pkl and feature_cols.pkl")

✔ Saved scaler.pkl and feature_cols.pkl
